# DataMate v2.5 (Scientific Verified Edition) — Qwen2.5-Coder-14B-Instruct (4-bit NF4)
**Kiến trúc 3 Trụ cột Tối ưu Điểm số Penalized Accuracy ($PA = \frac{\text{Đúng} - \text{Sai}}{N}$):**
1. **Tri thức Rulebook Động (`RulebookRetriever` — *BIRD Benchmark*, NeurIPS 2023):** Tự động nạp `/data/rulebook.md` và tiêm đúng các điều khoản/công thức toán học theo ngành hàng vào Prompt của SLM.
2. **Không gian Sinh code Nguyên tử (`Atomic Rulebook Helpers` + `Dynamic Few-Shot` — *Chain-of-Table* ICLR 2024 & *DAIL-SQL* VLDB 2024):** Cung cấp 25 hàm nguyên tử chuẩn Rulebook (`gross_profit`, `net_profit`, `peak_hour`, `busiest_date`, `loyal_customer`, `big_spender_customer`, `fickle_customer`, `popular_item`, `weekday_avg_revenue`, v.v.) trong Sandbox.
3. **Cổng Chấp nhận Kết quả Kép (`Gate G5 Entity Grounding` + `Gate G6 Dual-Perspective Execution Consistency` + `Kadavath P(True) Calibration` — *LEVER* ICML 2023, *CHESS* 2024, *Kadavath et al.* 2022):** Chỉ xuất đáp án của SLM khi hai góc nhìn sinh code độc lập thực thi ra **cùng một kết quả** (`results_agree == True`), thực thể có thật trong cửa hàng, và xác suất tự đánh giá $P(\text{True}) \ge 0.65$; nếu có bất kỳ nghi ngờ nào sẽ tự động `abstain` ($0$ điểm) để bảo vệ điểm số khỏi bị phạt $-1$.

In [ ]:
# 1. Cài đặt các thư viện cần thiết trên Google Colab GPU (L4 / A100 / T4)
# Lưu ý: Colab đã có sẵn torch, pandas==2.2.3, tqdm. Không dùng -U pandas để tránh xung đột với môi trường Colab.
!pip install -q -U transformers accelerate bitsandbytes peft rapidfuzz openpyxl


In [ ]:
# 2. Kiểm tra GPU & thiết lập Seed cố định (Deterministic 100% theo quy chế thi)
import os
import json
import time
import torch
from collections import Counter
from tqdm.auto import tqdm

os.environ["PYTHONHASHSEED"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f"GPU khả dụng: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Cảnh báo: Đang chạy trên CPU!")

In [ ]:
# 3. Giải nén gói mã nguồn main_src.zip và dữ liệu public_test.zip
import sys
import os

# 1. Tự động tìm và giải nén main_src.zip (xử lý chống lồng thư mục main_src/main_src)
zip_candidates = [f for f in os.listdir("/content") if "main_src" in f and f.endswith(".zip")]
if zip_candidates:
    zip_name = zip_candidates[0]
    !rm -rf /content/main_src /content/temp_extract
    !unzip -q -o "/content/{zip_name}" -d /content/temp_extract
    if os.path.exists("/content/temp_extract/main_src"):
        !mv /content/temp_extract/main_src /content/main_src
    else:
        !mv /content/temp_extract /content/main_src
    !rm -rf /content/temp_extract
    print(f"Đã giải nén thành công {zip_name} vào /content/main_src!")
elif os.path.exists("/content/drive/MyDrive/main_src.zip"):
    !rm -rf /content/main_src /content/temp_extract
    !unzip -q -o "/content/drive/MyDrive/main_src.zip" -d /content/temp_extract
    if os.path.exists("/content/temp_extract/main_src"):
        !mv /content/temp_extract/main_src /content/main_src
    else:
        !mv /content/temp_extract /content/main_src
    !rm -rf /content/temp_extract
    print("Đã giải nén main_src.zip từ Google Drive vào /content/main_src!")
else:
    print("CẢNH BÁO: Chưa tìm thấy file main_src.zip! Hãy kéo thả file main_src.zip vào thanh tệp bên trái Colab!")

# 2. Tự động tìm và giải nén public_test.zip nếu có
pt_candidates = [f for f in os.listdir("/content") if "public_test" in f and f.endswith(".zip")]
if pt_candidates:
    !unzip -q -o "/content/{pt_candidates[0]}" -d /content/public_test
    print(f"Đã giải nén {pt_candidates[0]} vào /content/public_test!")
elif os.path.exists("/content/drive/MyDrive/public_test.zip"):
    !unzip -q -o "/content/drive/MyDrive/public_test.zip" -d /content/public_test
    print("Đã giải nén public_test.zip từ Google Drive vào /content/public_test!")

for p in [os.path.abspath("."), "/content", "/content/main_src"]:
    if p not in sys.path:
        sys.path.insert(0, p)

from main_src.pipeline import DataMatePipeline
from main_src.formatters import dumps_submission_record, make_submission_record
from main_src.agent import AnswerSpecInferer, AnswerabilityGate
from main_src.sandbox import extract_code
print("ĐÃ NẠP THÀNH CÔNG GÓI MÃ NGUỒN main_src!")


In [ ]:
# 4. Cấu hình & Khởi tạo DataMatePipeline cho tập Public Test
DATA_ROOT = "/content/public_test"
# Tự động định vị thư mục chứa questions.jsonl của public test
for cand in ["/content/public_test", "/content/public_test/public_test", "/content/data", "/content"]:
    if os.path.exists(os.path.join(cand, "questions.jsonl")):
        DATA_ROOT = cand
        break

OUTPUT_SUBMISSION = "/content/submission.jsonl"
LOG_DIR = "/content/logs"
MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct"

USE_LLM = True                           # Bật Qwen2.5-Coder-14B-Instruct 4-bit
BATCH_SIZE = 8                           # Batch size tối ưu cho GPU L4 24GB VRAM
REQUIRE_DUAL_EXECUTION_AGREEMENT = True  # BẮT BUỘC: Góc nhìn A và Góc nhìn B phải ra cùng kết quả thực thi
ENABLE_P_TRUE_GATE = True                # BẮT BUỘC: Kiểm định xác suất đúng Logprob P(True) theo Kadavath et al.
P_TRUE_THRESHOLD = 0.65                  # Ngưỡng P(True) tối thiểu để chấp nhận câu trả lời của SLM

pipeline = DataMatePipeline(
    data_root_dir=DATA_ROOT,
    log_dir=LOG_DIR,
    use_llm=USE_LLM,
    llm_provider="transformers",
    llm_model=MODEL_ID,
    load_in_4bit=True,
    sandbox_timeout_s=10.0,
    require_dual_execution_agreement=REQUIRE_DUAL_EXECUTION_AGREEMENT,
    enable_p_true_gate=ENABLE_P_TRUE_GATE,
    p_true_threshold=P_TRUE_THRESHOLD,
)
print(f"Đã khởi tạo Pipeline cho Public Test tại: {DATA_ROOT}")
print(f"Rulebook loaded: {bool(pipeline.agent.rulebook_retriever.raw_rulebook_text)}")


In [ ]:
# 5. Thực thi Chu trình 4 Giai đoạn (Fast-Path -> SLM Round 1 -> SLM Round 2 -> Dual-Execution + P(True) Verification)
questions_file = os.path.join(DATA_ROOT, "questions.jsonl")
if os.path.exists(questions_file):
    t_start = time.time()
    results, gate_stats = pipeline.run_batch_inference(
        questions_path=questions_file,
        output_submission_path=OUTPUT_SUBMISSION,
        batch_size=BATCH_SIZE,
    )
    elapsed_min = (time.time() - t_start) / 60.0
    type_counts = Counter(r["type"] for r in results)
    print("\n" + "=" * 65)
    print(f"HOÀN TẤT TRONG {elapsed_min:.2f} PHÚT! TỔNG SỐ CÂU: {len(results)}")
    print("-" * 65)
    print("1. Thống kê theo Kiểu câu trả lời (type):")
    for k, v in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"   - {k:15s}: {v:4d} ({v / len(results) * 100:.1f}%)")
    print("-" * 65)
    print("2. Thống kê chi tiết theo Cổng Kiểm định (Gate Breakdown):")
    for g, cnt in sorted(gate_stats.items(), key=lambda x: -x[1]):
        print(f"   - {g:40s}: {cnt:4d}")
    print("=" * 65)
else:
    print(f"Không tìm thấy file {questions_file}. Hãy kiểm tra lại đường dẫn DATA_ROOT!")

In [ ]:
# 6. (Tùy chọn) Chấm điểm trực tiếp nếu chạy thử trên tập Train (có file answers.jsonl)
answers_file = os.path.join(DATA_ROOT, "answers.jsonl")
if os.path.exists(answers_file) and os.path.exists(OUTPUT_SUBMISSION):
    from main_src.run_local_test import score_predictions
    with open(answers_file, "r", encoding="utf-8") as f:
        gold_map = {json.loads(line)["id"]: json.loads(line) for line in f if line.strip()}
    with open(OUTPUT_SUBMISSION, "r", encoding="utf-8") as f:
        preds = [json.loads(line) for line in f if line.strip()]
    stats = score_predictions(preds, gold_map)
    print("=== KẾT QUẢ CHẤM ĐIỂM TRÊN TẬP CÓ ĐÁP ÁN ===")
    print(json.dumps(stats, ensure_ascii=False, indent=2))

In [ ]:
# 7. Tải file submission.jsonl về máy tính cá nhân để nộp ban tổ chức
if os.path.exists(OUTPUT_SUBMISSION):
    try:
        from google.colab import files
        files.download(OUTPUT_SUBMISSION)
        print(f"Đang tải {OUTPUT_SUBMISSION} về máy...")
    except Exception as e:
        print(f"File kết quả sẵn sàng tại: {OUTPUT_SUBMISSION}")
else:
    print("Chưa có file submission.jsonl để tải!")
